In [1]:
import pandas as pd
import duckdb as dck
filepath=r"C:\Users\janek\OneDrive\Pulpit\pythoncode\reader\countries.csv"


df = pd.read_csv(filepath)

print(df.head())

            name            capital  population     area     region
0  South Georgia  King Edward Point          30   3903.0  Antarctic
1        Grenada       St. George's      112519    344.0   Americas
2    Switzerland               Bern     8654622  41284.0     Europe
3   Sierra Leone           Freetown     7976985  71740.0     Africa
4        Hungary           Budapest     9749763  93028.0     Europe


In [2]:
result = dck.sql("""
    SELECT *
    FROM read_csv_auto('countries.csv')
    LIMIT 5
""").df()

result

,name,capital,population,area,region
0,South Georgia,King Edward Point,30,3903.0,Antarctic
1,Grenada,St. George's,112519,344.0,Americas
2,Switzerland,Bern,8654622,41284.0,Europe
3,Sierra Leone,Freetown,7976985,71740.0,Africa
4,Hungary,Budapest,9749763,93028.0,Europe


In [3]:

dck.sql("""
    CREATE TABLE countries AS
    SELECT * FROM read_csv_auto('countries.csv')
""")

result = dck.sql("""
    SELECT *
    FROM countries
    LIMIT 5
""").df()

result

,name,capital,population,area,region
0,South Georgia,King Edward Point,30,3903.0,Antarctic
1,Grenada,St. George's,112519,344.0,Americas
2,Switzerland,Bern,8654622,41284.0,Europe
3,Sierra Leone,Freetown,7976985,71740.0,Africa
4,Hungary,Budapest,9749763,93028.0,Europe


In [4]:
# Task1 Largest population 

query1="""SELECT name, population From countries ORDER BY population DESC LIMIT 5
"""

query1b="""SELECT name, population From countries where population=(select max(population) FROM countries) 
"""

task1b=dck.sql(query1b)

task1b

nameVARCHAR,populationBIGINT
China,1402112000


In [9]:
# Task2 min area by region 

query2="""SELECT region,  min(area)
FROM countries
GROUP BY region
"""

query2b="""SELECT region, name, area
FROM (
    SELECT region, name, area,
           MIN(area) OVER (
               PARTITION BY region
           ) AS armin
    FROM countries
) AS ranked
WHERE area = armin
"""

task2=dck.sql(query2)
task2
task2b=dck.sql(query2b)
task2b


regionVARCHAR,nameVARCHAR,areaDOUBLE
Americas,Saint Barthélemy,21.0
Oceania,Tokelau,12.0
Antarctic,Bouvet Island,49.0
Asia,Macau,30.0
Europe,Vatican City,0.44
Africa,British Indian Ocean Territory,60.0


In [14]:
# make density and pick higest

query3="""SELECT name, density
from (SELECT name, ROUND((population/area::NUMERIC),2) as density
from countries where area>0)
ORDER BY density DESC
LIMIT 5
"""
task3=dck.sql(query3)
task3

nameVARCHAR,densityDOUBLE
Macau,21644.73
Monaco,19427.72
Singapore,8008.18
Hong Kong,6794.11
Gibraltar,5615.17


In [19]:
# mean population per country in region, but only from countries with area >100

query4="""SELECT region, round(avg(population),3) as avg_population FROM countries
WHERE area>100
GROUP BY region
"""

task4=dck.sql(query4)
task4

regionVARCHAR,avg_populationDOUBLE
Antarctic,357.5
Asia,93958074.122
Oceania,2052023.476
Europe,15557578.104
Americas,20416300.78
Africa,23484348.879


In [23]:
# task7 3 largest countries in area 

query5=""" SELECT region, name, area
FROM (select *, row_number()  over(partition by region order by area desc) as rn from countries)  
where rn<=3
ORDER BY region ASC, area DESC
"""

task5=dck.sql(query5)
task5


regionVARCHAR,nameVARCHAR,areaDOUBLE
Africa,Algeria,2381741.0
Africa,DR Congo,2344858.0
Africa,Sudan,1886068.0
Americas,Canada,9984670.0
Americas,United States,9372610.0
Americas,Brazil,8515767.0
Antarctic,Antarctica,14000000.0
Antarctic,French Southern and Antarctic Lands,7747.0
Antarctic,South Georgia,3903.0
Asia,China,9706961.0


In [24]:
# task8 how many countries in each region

query6=""" SELECT region, count(name) FROM countries GROUP BY region 
"""
task6=dck.sql(query6)
task6


regionVARCHAR,"count(""name"")BIGINT"
Americas,56
Antarctic,5
Oceania,27
Europe,53
Africa,59
Asia,50


In [28]:
# task9 Find the median population of countries in each region.

query7=""" SELECT region, min(population), max(population), round(MEDIAN(population),2)
FROM countries GROUP BY region 
"""
task7=dck.sql(query7)
task7


regionVARCHAR,min(population)BIGINT,max(population)BIGINT,"round(median(population), 2)DOUBLE"
Oceania,56,25687041,59194.0
Antarctic,0,1000,30.0
Americas,300,329484123,398876.5
Europe,451,144104080,4047200.0
Asia,437483,1402112000,17109814.0
Africa,3000,206139587,11890781.0


In [32]:
#Task 10: Find countries whose population density exceeds the global median density. and population is above 1mln 

query8="""
WITH countdens as (
    SELECT *, round(population/area::numeric ,2) as density from countries 
)
SELECT name, population, density from countdens where population>1000000 and density>(select median(density) from countdens)
ORDER BY population desc
limit 10

"""

task8=dck.sql(query8)
task8

nameVARCHAR,populationBIGINT,densityDOUBLE
China,1402112000,144.44
India,1380004385,419.76
Indonesia,273523621,143.61
Pakistan,220892331,250.47
Nigeria,206139587,223.15
Bangladesh,164689383,1116.01
Japan,125836021,332.96
Ethiopia,114963583,104.11
Philippines,109581085,320.08
Egypt,102334403,102.08


In [41]:

query9 = """
SET VARIABLE globalpopsum = (
    SELECT SUM(population) FROM countries
);

SELECT
    region,
    ROUND(
        SUM(population) / getvariable('globalpopsum') * 100,
        2
    ) AS population_perc
FROM countries
GROUP BY region
ORDER BY population_perc DESC
"""

task9 = dck.sql(query9)
task9


regionVARCHAR,population_percDOUBLE
Asia,59.2
Africa,17.51
Americas,13.13
Europe,9.6
Oceania,0.55
Antarctic,0.0
